In [0]:
dbutils.widgets.text("date", "")
stat_date = dbutils.widgets.get("date").strip() or str(spark.sql("SELECT MAX(load_date) FROM workspace.steam.bronze_steam").first()[0])
spark.sql("USE workspace.steam")

display(spark.sql(f"""
WITH today  AS (SELECT appid FROM bronze_steam WHERE load_date = DATE'{stat_date}'),
     before AS (SELECT DISTINCT appid FROM bronze_steam WHERE load_date < DATE'{stat_date}')
SELECT
  (SELECT COUNT(*) FROM today)                                              AS bronze_rows_for_date,
  (SELECT COUNT(*) FROM today t LEFT ANTI JOIN before b ON t.appid = b.appid) AS new_games,
  (SELECT COUNT(*) FROM today t LEFT SEMI JOIN before b ON t.appid = b.appid) AS changed_games,
  (SELECT COUNT(*) FROM bronze_steam)                                       AS bronze_total,
  (SELECT COUNT(*) FROM silver_steam)                                       AS silver_total,
  (SELECT COUNT(*) FROM silver_steam WHERE first_load_date = DATE'{stat_date}') AS silver_new_for_date,
  (SELECT COUNT(*) FROM silver_steam
    WHERE last_load_date = DATE'{stat_date}' AND first_load_date < DATE'{stat_date}') AS silver_updated_for_date,
  (SELECT COUNT(*) FROM (SELECT appid FROM silver_steam GROUP BY appid HAVING COUNT(*) > 1)) AS silver_duplicate_appids
"""))